# 3 · Interpretabilidade e aplicação estratégica

Duas perguntas: **por que** o modelo decide o que decide, e **o que fazer** com isso.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

# O acesso ao BigQuery exige o token de curta duração:
#   export GCP_ACCESS_TOKEN=$(gcloud auth print-access-token)

import joblib
from src.common.config import MODELS_DIR
from src.modeling.dataset import carregar_abt, split_temporal
from src.preprocessing.features import ALVO, descartar_degeneradas, selecionar_features
from src.evaluation import interpret, aplicacao
from src.visualization import eda, plots

df = carregar_abt()
treino, teste = split_temporal(df)
features = selecionar_features(list(df.columns), permitir_defasadas=False)
features, _ = descartar_degeneradas(treino, features)
pipeline = joblib.load(MODELS_DIR / "modelo_temporal.joblib")
type(pipeline.named_steps["modelo"]).__name__

## Quatro leituras de importância

Elas têm vieses diferentes de propósito. Quando discordam, a discordância informa:
importância nativa alta com permutação baixa indica variável usada para ajustar ruído.

In [ ]:
interpret.importancia_nativa(pipeline, top=20)

In [ ]:
permut = interpret.importancia_permutacao(pipeline, teste[features],
                                          teste[ALVO].astype(int).to_numpy())
permut.head(20)

### SHAP

O único método que responde *"por que ESTE aluno foi classificado como em risco?"* —
a pergunta do gestor diante de uma lista de prioridades.

In [ ]:
valores, X_t, esperado = interpret.valores_shap(pipeline, teste[features])
plots.fig_shap_resumo(valores, X_t)

### Ablação por bloco

A leitura que sobrevive à multicolinearidade: variáveis que medem a mesma coisa
(IVS, renda, IDHM) saem juntas, então a queda observada é contribuição real.

In [ ]:
from src.modeling.candidatos import obter
from src.modeling.train import amostrar
est, escalonar, _ = obter("lightgbm")
ablacao = interpret.ablacao_por_bloco(amostrar(treino, 400_000), teste, features,
                                      est, escalonar=escalonar)
ablacao

## Da probabilidade individual à decisão pública

A predição individual tem teto baixo — o desfecho de uma criança depende de fatores
que nenhum dado público municipal captura. Mas o erro individual se cancela na média,
e é a média municipal que orienta a política.

In [ ]:
prob = pipeline.predict_proba(teste[features])[:, 1]
ranking = aplicacao.ranking_risco_municipal(teste, prob)
calib = aplicacao.calibracao_agregada(ranking)
calib

In [ ]:
plots.fig_calibracao_municipal(ranking, calib)

### A coluna acionável: o resíduo

O risco absoluto em geral só reflete a pobreza do território. O **resíduo**
(observado − previsto) isola o que o contexto não explica: muito negativo aponta
problema de gestão; muito positivo, prática que merece ser estudada.

In [ ]:
plots.fig_residuos(ranking)

In [ ]:
ranking[ranking["n_alunos"] >= 100].nsmallest(10, "residuo")[
    ["municipio", "uf", "n_alunos", "taxa_observada", "taxa_prevista", "residuo"]]

## Regiões com padrões semelhantes

Agrupamento por **contexto**, com o alvo deliberadamente de fora — assim a taxa de
alfabetização de cada grupo é um resultado da análise, não o critério que a produziu.

In [ ]:
mun = eda.agregar_municipio(teste)
municipios, perfil = aplicacao.agrupar_municipios(mun, k=5)
perfil[["n_municipios", "taxa_observada"]]

In [ ]:
plots.fig_grupos(perfil, municipios)

## Municípios em risco de não atingir a meta

A taxa municipal prevista é a média de indicadores de Bernoulli; sua distribuição
amostral é aproximadamente normal, e a probabilidade de ficar abaixo da meta é
Φ((meta − previsto) / erro-padrão).

**Ressalva:** o cálculo supõe independência condicional entre alunos. Como colegas de
escola compartilham choques não observados, o erro-padrão real é maior — a ordenação é
confiável, a magnitude não.

In [ ]:
# Requer o desenho espacial (as metas são features defasadas, ausentes em 2023).
# Ver reports/aplicacao_espacial.md